In [ ]:
GITHUB_USERNAME = "Your_github_id"
GITHUB_TOKEN = "Your_github_token"#guide will be given in the redme file 

In [13]:
import requests
import pandas as pd
BASE_URL = "https://api.github.com"
HEADERS = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "X-GitHub-Api-Version": "2022-11-28"}
def get_all(url, params=None):
    params = params or {}
    params["per_page"] = 100
    page = 1
    data = []
    while True:
        params["page"] = page
        r = requests.get(url, headers=HEADERS, params=params)
        r.raise_for_status()
        x = r.json()
        if not x:
            break
        if isinstance(x, dict):
            return x
        data.extend(x)
        if len(x) < 100:
            break
        page += 1
    return data
# A Profile
u = requests.get(
    f"{BASE_URL}/users/{GITHUB_USERNAME}",
    headers=HEADERS
).json()
df_user = pd.DataFrame([u])
# B. Repositories
repos = get_all(
    f"{BASE_URL}/users/{GITHUB_USERNAME}/repos",
    {"type": "owner", "sort": "updated"})
df_repositories = pd.DataFrame(repos)
traffic = []
languages = []
for repo in repos:
    owner = repo["owner"]["login"]
    name = repo["name"]
    repo_id = repo["id"]
    # Traffic
    try:
        views = requests.get(
            f"{BASE_URL}/repos/{owner}/{name}/traffic/views",
            headers=HEADERS
        ).json().get("views", [])
        clones = requests.get(
            f"{BASE_URL}/repos/{owner}/{name}/traffic/clones",
            headers=HEADERS
        ).json().get("clones", [])
        view_map = {x["timestamp"][:10]: x for x in views}
        clone_map = {x["timestamp"][:10]: x for x in clones}
        for date in sorted(set(view_map) | set(clone_map)):
            v = view_map.get(date, {})
            c = clone_map.get(date, {})
            traffic.append({
                "repo_id": repo_id,
                "repo_name": name,
                "date": date,
                "views": v.get("count", 0),
                "unique_visitors": v.get("uniques", 0),
                "clones": c.get("count", 0),
                "unique_cloners": c.get("uniques", 0)})
    except:
        pass
    # Languages
    try:
        lang = requests.get(
            f"{BASE_URL}/repos/{owner}/{name}/languages",
            headers=HEADERS
        ).json()
        for language, bytes_count in lang.items():
            languages.append({
                "repo_id": repo_id,
                "repo_name": name,
                "language": language,
                "bytes": bytes_count})
    except:
        pass
df_traffic = pd.DataFrame(traffic)
df_languages = pd.DataFrame(languages)

In [14]:
df_user.drop(columns=[
    'id',
    'node_id',
    'avatar_url',
    'gravatar_id',
    'url',
    'html_url',
    'followers_url',
    'following_url',
    'gists_url',
    'starred_url',
    'subscriptions_url',
    'organizations_url',
    'repos_url',
    'events_url',
    'received_events_url',
    'type',
    'user_view_type',
    'site_admin',
    'blog',
    'email',
    'bio',
    'twitter_username',
    'private_gists',
    'total_private_repos',
    'owned_private_repos',
    'disk_usage',
    'collaborators',
    'plan'
], inplace=True)

In [15]:
df_repositories.drop(columns=[
    'node_id',
    'owner',
    'html_url',
    'url',
    'forks_url',
    'keys_url',
    'collaborators_url',
    'teams_url',
    'hooks_url',
    'issue_events_url',
    'events_url',
    'assignees_url',
    'branches_url',
    'tags_url',
    'blobs_url',
    'git_tags_url',
    'git_refs_url',
    'trees_url',
    'statuses_url',
    'languages_url',
    'stargazers_url',
    'contributors_url',
    'subscribers_url',
    'subscription_url',
    'commits_url',
    'git_commits_url',
    'comments_url',
    'issue_comment_url',
    'contents_url',
    'compare_url',
    'merges_url',
    'archive_url',
    'downloads_url',
    'issues_url',
    'pulls_url',
    'milestones_url',
    'notifications_url',
    'labels_url',
    'git_url',
    'ssh_url',
    'clone_url',
    'svn_url',
    'homepage',
    'mirror_url',
    'permissions',
    'forks',
    'watchers',
    'open_issues'
], inplace=True)

In [ ]:
import sqlite3
import json

def prepare_for_sqlite(df):
    df = df.copy()
    for col in df.columns:
        df[col] = df[col].apply(
            lambda x: json.dumps(x, ensure_ascii=False) 
            if isinstance(x, (dict, list)) 
            else x
        )
    return df

conn = sqlite3.connect("github_analytics.db")

tables = {
    "user": df_user,
    "repositories": df_repositories,
    "traffic": df_traffic,
    "languages": df_languages
}

for name, df in tables.items():
    df_sql = prepare_for_sqlite(df)
    df_sql.to_sql(name, conn, if_exists="replace", index=False)

conn.close()

user → 1 rows stored
repositories → 17 rows stored
traffic → 238 rows stored
languages → 12 rows stored

Done! All DataFrames stored in SQLite.
